Clean and standardize raw shade data into an analytics-ready view with RGB conversions and classification flags.
*Co-authored with CoCo*

In [ ]:
%%sql -r dataframe_1
USE DATABASE MARCJACOBS_BEAUTY_REVIVAL;

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE VIEW MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.CLEAN_SHADE_MASTER AS
SELECT
  LOWER(TRIM(s.era)) AS era,
  s.launch_year::INT AS launch_year,
  INITCAP(TRIM(s.category)) AS category,
  INITCAP(TRIM(s.product_type)) AS product_type,
  TRIM(s.product_name) AS product_name,
  INITCAP(TRIM(p.formula)) AS formula,
  INITCAP(TRIM(s.finish)) AS finish,
  LOWER(TRIM(s.finish_group)) AS finish_group,
  TRIM(s.shade_code) AS shade_code,
  s.pan_number,
  TRIM(s.shade_name) AS shade_name,
  TRIM(s.shade_description) AS shade_description,
  LOWER(TRIM(s.color_family)) AS color_family,
  s.shade_depth_order,
  LOWER(NULLIF(TRIM(s.shade_depth_group), '')) AS shade_depth_group,
  LOWER(NULLIF(TRIM(s.undertone_inferred_cmyk), '')) AS undertone_inferred_cmyk,
  LOWER(NULLIF(TRIM(s.undertone_confidence), '')) AS undertone_confidence,
  LOWER(NULLIF(TRIM(s.shade_function), '')) AS shade_function,

  -- Price and size from PRODUCTS
  p.price_usd::FLOAT AS sticker_price,
  p.size_oz::FLOAT AS size_oz,
  CASE WHEN p.size_oz IS NOT NULL AND p.size_oz > 0
       THEN p.price_usd::FLOAT / p.size_oz::FLOAT
       ELSE NULL END AS price_per_oz,

  -- CMYK values
  s.c_pct::FLOAT AS c_pct,
  s.m_pct::FLOAT AS m_pct,
  s.y_pct::FLOAT AS y_pct,
  s.k_pct::FLOAT AS k_pct,

  -- CMYK converted to RGB percentage scale (0 to 1)
  ((1 - s.c_pct::FLOAT / 100) * (1 - s.k_pct::FLOAT / 100)) AS r_pct,
  ((1 - s.m_pct::FLOAT / 100) * (1 - s.k_pct::FLOAT / 100)) AS g_pct,
  ((1 - s.y_pct::FLOAT / 100) * (1 - s.k_pct::FLOAT / 100)) AS b_pct,

  -- RGB converted to 0-255 scale
  ROUND(((1 - s.c_pct::FLOAT / 100) * (1 - s.k_pct::FLOAT / 100)) * 255) AS r_255,
  ROUND(((1 - s.m_pct::FLOAT / 100) * (1 - s.k_pct::FLOAT / 100)) * 255) AS g_255,
  ROUND(((1 - s.y_pct::FLOAT / 100) * (1 - s.k_pct::FLOAT / 100)) * 255) AS b_255,

  s.color_value_source,

  -- Classification flags
  CASE WHEN LOWER(s.product_type) IN ('foundation', 'concealer', 'powder foundation', 'bronzer')
       THEN TRUE ELSE FALSE END AS is_complexion,
  CASE WHEN LOWER(s.category) IN ('face', 'eyes', 'lips')
       THEN TRUE ELSE FALSE END AS is_color_cosmetic,
  CASE WHEN LOWER(s.product_name) LIKE '%joystick%'
        OR LOWER(s.shade_description) LIKE '%lip and cheek%'
       THEN TRUE ELSE FALSE END AS is_multiuse,
  CASE WHEN LOWER(p.formula) LIKE '%powder%'
       THEN TRUE ELSE FALSE END AS is_powder,
  CASE WHEN LOWER(p.formula) LIKE '%cream%'
       THEN TRUE ELSE FALSE END AS is_cream,

  -- Out of stock
  COALESCE(s.is_oos, FALSE) AS is_out_of_stock,
  s.oos_capture_date,

  -- Shade count per product within era
  COUNT(*) OVER (PARTITION BY s.era, s.product_name) AS shade_count_for_product,

  s.source_url

FROM MARCJACOBS_BEAUTY_REVIVAL.RAW.SHADES_RAW s
LEFT JOIN MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.PRODUCTS p
  ON LOWER(TRIM(s.era)) = LOWER(TRIM(p.era))
 AND TRIM(s.product_name) = TRIM(p.product_name);